# NB07b — nnU-Net Stage 1 Retrain (80/20 Split)

**Purpose:** Retrain nnU-Net kidney localizer using the 80/20 stratified split.  
**Preprocessing:** Already done for all 208 cases. Only splits_final.json needs updating.  
**Training:** 3d_lowres fold 0 only. Try 3d_fullres if time permits.  
**After training:** Predict all 208 cases → NB07c extracts ROIs.

## Run Order:
1. Cell 1: Setup + verify paths
2. Cell 2: Override splits_final.json with 80/20 split
3. Cell 3: Train nnU-Net (blocks for 6-12 hours, writes to log file)
4. Cell 3-Monitor: Check training progress (run in separate notebook)
5. Cell 4: Predict all 208 cases
6. Cell 5: Verify predictions
7. Cell 6: Check training log

**⚠️ IMPORTANT:** Cell 3 runs training via subprocess. If Jupyter disconnects, 
training may stop. For safety, you can copy the printed command from Cell 3 
and run it directly in CMD instead. The cell prints the exact command before starting.

In [1]:
# ======== Cell 1: Setup & Verify ========
import os, json
from pathlib import Path

# nnU-Net paths
NNUNET_BASE = r"D:\fsdf_uca_kits\nnunet"
os.environ['nnUNet_raw'] = os.path.join(NNUNET_BASE, 'raw')
os.environ['nnUNet_preprocessed'] = os.path.join(NNUNET_BASE, 'preprocessed')
os.environ['nnUNet_results'] = os.path.join(NNUNET_BASE, 'results')

# Project paths
OUTPUT_DIR = Path(r"D:\fsdf_uca_kits\outputs")
PREPROC_DIR = OUTPUT_DIR / "preprocessed"
SPLIT_80_20 = OUTPUT_DIR / "stats" / "fold_splits_80_20.json"
NNUNET_PP = Path(os.environ['nnUNet_preprocessed']) / "Dataset001_KiTS"

# Verify everything exists
checks = {
    "80/20 split file": SPLIT_80_20,
    "nnU-Net preprocessed": NNUNET_PP,
    "nnU-Net plans": NNUNET_PP / "nnUNetPlans.json",
    "nnU-Net splits_final": NNUNET_PP / "splits_final.json",
    "3d_lowres data": NNUNET_PP / "nnUNetPlans_3d_lowres",
    "3d_fullres data": NNUNET_PP / "nnUNetPlans_3d_fullres",
}

print("Path verification:")
all_ok = True
for name, path in checks.items():
    exists = path.exists()
    if not exists: all_ok = False
    print(f"  {'✓' if exists else '✗'} {name}: {path}")

# Load 80/20 split
with open(SPLIT_80_20) as f:
    split_data = json.load(f)
train_cases = sorted(split_data['folds']['fold_0']['train'])
val_cases = sorted(split_data['folds']['fold_0']['val'])
print(f"\n80/20 split: {len(train_cases)} train, {len(val_cases)} val")

# Show current splits_final.json
with open(NNUNET_PP / "splits_final.json") as f:
    current_split = json.load(f)
print(f"Current nnU-Net fold 0: {len(current_split[0]['train'])} train, {len(current_split[0]['val'])} val")

if all_ok:
    print("\n✓ All paths verified. Ready to override split.")
else:
    print("\n✗ Some paths missing! Fix before proceeding.")

Path verification:
  ✓ 80/20 split file: D:\fsdf_uca_kits\outputs\stats\fold_splits_80_20.json
  ✓ nnU-Net preprocessed: D:\fsdf_uca_kits\nnunet\preprocessed\Dataset001_KiTS
  ✓ nnU-Net plans: D:\fsdf_uca_kits\nnunet\preprocessed\Dataset001_KiTS\nnUNetPlans.json
  ✓ nnU-Net splits_final: D:\fsdf_uca_kits\nnunet\preprocessed\Dataset001_KiTS\splits_final.json
  ✓ 3d_lowres data: D:\fsdf_uca_kits\nnunet\preprocessed\Dataset001_KiTS\nnUNetPlans_3d_lowres
  ✓ 3d_fullres data: D:\fsdf_uca_kits\nnunet\preprocessed\Dataset001_KiTS\nnUNetPlans_3d_fullres

80/20 split: 166 train, 42 val
Current nnU-Net fold 0: 138 train, 70 val

✓ All paths verified. Ready to override split.


In [2]:
# ======== Cell 2: Override splits_final.json with 80/20 split ========
# This overwrites nnU-Net's fold 0 to match your 80/20 stratified split.
# IMPORTANT: This changes the training set from 138 → 166 cases.

import json
from pathlib import Path

NNUNET_PP = Path(os.environ['nnUNet_preprocessed']) / "Dataset001_KiTS"
splits_path = NNUNET_PP / "splits_final.json"

# Backup old split
import shutil
backup_path = NNUNET_PP / "splits_final_old_70_30.json"
if not backup_path.exists():
    shutil.copy2(splits_path, backup_path)
    print(f"Backed up old split → {backup_path.name}")
else:
    print(f"Backup already exists: {backup_path.name}")

# Convert case names to nnU-Net format
train_ids = [f"KiTS_{c.replace('case_', '')}" for c in train_cases]
val_ids = [f"KiTS_{c.replace('case_', '')}" for c in val_cases]
all_ids = sorted(set(train_ids + val_ids))

# Verify all IDs exist in preprocessed data
existing_ids = set()
for config_dir in ['nnUNetPlans_3d_lowres', 'nnUNetPlans_3d_fullres']:
    config_path = NNUNET_PP / config_dir
    if config_path.exists():
        for f in config_path.iterdir():
            if f.suffix == '.npz':
                existing_ids.add(f.stem)

missing_train = [t for t in train_ids if t not in existing_ids]
missing_val = [v for v in val_ids if v not in existing_ids]
print(f"Preprocessed cases found: {len(existing_ids)}")
if missing_train:
    print(f"  WARNING: {len(missing_train)} train cases missing from preprocessed!")
    print(f"  Missing: {missing_train[:5]}")
if missing_val:
    print(f"  WARNING: {len(missing_val)} val cases missing from preprocessed!")

# Build new splits_final.json
nnunet_splits = []
nnunet_splits.append({"train": train_ids, "val": val_ids})  # Fold 0: 80/20
for i in range(1, 5):  # Folds 1-4: dummy
    nnunet_splits.append({"train": all_ids, "val": []})

with open(splits_path, 'w') as f:
    json.dump(nnunet_splits, f, indent=2)

# Verify
with open(splits_path) as f:
    saved = json.load(f)
print(f"\nNew splits_final.json written:")
print(f"  Fold 0 train: {len(saved[0]['train'])} cases")
print(f"  Fold 0 val:   {len(saved[0]['val'])} cases")
print(f"  Overlap:      {len(set(saved[0]['train']) & set(saved[0]['val']))} (must be 0)")
print(f"  Total:        {len(set(saved[0]['train']) | set(saved[0]['val']))}")

# Check excluded cases are NOT in split
excluded_ids = {'KiTS_00015', 'KiTS_00037'}
in_train = excluded_ids & set(saved[0]['train'])
in_val = excluded_ids & set(saved[0]['val'])
if in_train or in_val:
    print(f"  ⚠️ EXCLUDED CASES FOUND: train={in_train}, val={in_val}")
else:
    print(f"  ✓ Excluded cases (15, 37) correctly absent")

print(f"\n✓ Ready to train. Run Cell 3 for CMD commands.")

Backed up old split → splits_final_old_70_30.json
Preprocessed cases found: 0
  Missing: ['KiTS_00001', 'KiTS_00002', 'KiTS_00003', 'KiTS_00004', 'KiTS_00005']

New splits_final.json written:
  Fold 0 train: 166 cases
  Fold 0 val:   42 cases
  Overlap:      0 (must be 0)
  Total:        208
  ✓ Excluded cases (15, 37) correctly absent

✓ Ready to train. Run Cell 3 for CMD commands.


In [ ]:
# ======== Cell 3: Train nnU-Net from Jupyter ========
# Writes output to log file. Monitor with Cell 3-Monitor.
# IMPORTANT: Close all other notebooks using GPU first!
#
# ⚠️ SAFER ALTERNATIVE: Copy the command printed below and run in CMD.
#    If Jupyter disconnects, CMD training continues.

import subprocess, time, os, shutil
from pathlib import Path

CONFIGURATION = "3d_lowres"  # Change to "3d_fullres" if you want (may OOM)
FOLD = 0
LOG_FILE = Path(r"D:\fsdf_uca_kits\outputs\nnunet_train_80_20.log")

# --- BACKUP old NB06 model before overwriting ---
OLD_MODEL = Path(os.environ['nnUNet_results']) / "Dataset001_KiTS" / \
            f"nnUNetTrainer__nnUNetPlans__{CONFIGURATION}" / "fold_0"
BACKUP_MODEL = Path(os.environ['nnUNet_results']) / "Dataset001_KiTS" / \
               f"nnUNetTrainer__nnUNetPlans__{CONFIGURATION}" / "fold_0_backup_old_split"

if OLD_MODEL.exists() and not BACKUP_MODEL.exists():
    shutil.copytree(OLD_MODEL, BACKUP_MODEL)
    print(f"✓ Backed up old model → {BACKUP_MODEL.name}/")
elif BACKUP_MODEL.exists():
    print(f"✓ Backup already exists: {BACKUP_MODEL.name}/")
else:
    print("No existing model to back up (first training)")

print(f"\nTraining: Dataset001_KiTS, {CONFIGURATION}, fold {FOLD}")
print(f"Train cases: {len(train_cases)}, Val cases: {len(val_cases)}")
print(f"Log file: {LOG_FILE}")

# Print CMD alternative
print(f"""
========================================
ALTERNATIVE: Run this in CMD instead
========================================
set nnUNet_raw=D:\\fsdf_uca_kits\\nnunet\\raw
set nnUNet_preprocessed=D:\\fsdf_uca_kits\\nnunet\\preprocessed
set nnUNet_results=D:\\fsdf_uca_kits\\nnunet\\results
conda activate C:\\Users\\vejey\\anaconda3\\envs\\kidney
nnUNetv2_train 001 {CONFIGURATION} {FOLD} --c --npz -device cuda
========================================
""")

print("Starting training... (this cell blocks for 6-12 hours)")
print("Monitor progress: run Cell 3-Monitor in a separate notebook\n")

t0 = time.time()

with open(LOG_FILE, 'w') as log:
    process = subprocess.Popen(
        ["nnUNetv2_train", "001", CONFIGURATION, str(FOLD),
         "--c",      # resume from checkpoint if interrupted
         "--npz", "-device", "cuda"],
        stdout=log, stderr=subprocess.STDOUT,
        env={**os.environ,
             'nnUNet_raw': r'D:\fsdf_uca_kits\nnunet\raw',
             'nnUNet_preprocessed': r'D:\fsdf_uca_kits\nnunet\preprocessed',
             'nnUNet_results': r'D:\fsdf_uca_kits\nnunet\results'},
        text=True
    )
    process.wait()

hours = (time.time() - t0) / 3600
print(f"\nTraining completed in {hours:.1f} hours")
print(f"Return code: {process.returncode}")

# Show last 20 lines of log
with open(LOG_FILE) as f:
    lines = f.readlines()
print(f"\nLast 20 lines of log:")
for line in lines[-20:]:
    print(f"  {line.rstrip()}")

In [ ]:
# ======== Cell 3-Monitor: Check Training Progress (run in separate notebook) ========
# Run this in a SEPARATE Jupyter tab/notebook while Cell 3 is running.
# Re-run this cell every few minutes to see progress.

from pathlib import Path

LOG_FILE = Path(r"D:\fsdf_uca_kits\outputs\nnunet_train_80_20.log")

if not LOG_FILE.exists():
    print("Log file not found. Is training running (Cell 3)?")
else:
    with open(LOG_FILE) as f:
        lines = f.readlines()
    
    print(f"Log: {len(lines)} lines")
    
    # Find epoch info
    epoch_lines = [l for l in lines if 'Epoch' in l or 'epoch' in l or 'EMA' in l]
    if epoch_lines:
        print(f"\nLatest epochs:")
        for l in epoch_lines[-10:]:
            print(f"  {l.rstrip()}")
    
    # Find best info
    best_lines = [l for l in lines if 'best' in l.lower()]
    if best_lines:
        print(f"\nBest so far:")
        for l in best_lines[-3:]:
            print(f"  {l.rstrip()}")
    
    print(f"\nLast 5 lines:")
    for l in lines[-5:]:
        print(f"  {l.rstrip()}")

In [ ]:
# ======== Cell 4: Predict all 208 cases ========
# Run AFTER training completes (Cell 3).

import subprocess, time, os, shutil
from pathlib import Path

CONFIGURATION = "3d_lowres"  # Must match what you trained
FOLD = 0

# Input folder (reuse existing NIfTI images)
INPUT_FOLDER = OUTPUT_DIR / "stage1_nnunet_predictions" / "input"
OUTPUT_FOLDER = OUTPUT_DIR / "stage1_nnunet_80_20" / "output"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Verify input exists
n_input = len(list(INPUT_FOLDER.glob("*.nii.gz"))) if INPUT_FOLDER.exists() else 0
print(f"Input images: {n_input}")

if n_input == 0:
    print("ERROR: No input images found!")
    print(f"Expected at: {INPUT_FOLDER}")
    print("Run NB06 Cell 2 first to convert data, or check path.")
else:
    print(f"Output folder: {OUTPUT_FOLDER}")
    print(f"Predicting with {CONFIGURATION}, fold {FOLD}...")
    print(f"(Takes ~10-20 minutes)\n")
    
    t0 = time.time()
    LOG_FILE = OUTPUT_DIR / "nnunet_predict_80_20.log"
    
    with open(LOG_FILE, 'w') as log:
        process = subprocess.Popen(
            ["nnUNetv2_predict",
             "-i", str(INPUT_FOLDER),
             "-o", str(OUTPUT_FOLDER),
             "-d", "001",
             "-c", CONFIGURATION,
             "-f", str(FOLD),
             "--disable_tta",
             "-device", "cuda"],
            stdout=log, stderr=subprocess.STDOUT,
            env={**os.environ,
                 'nnUNet_raw': r'D:\fsdf_uca_kits\nnunet\raw',
                 'nnUNet_preprocessed': r'D:\fsdf_uca_kits\nnunet\preprocessed',
                 'nnUNet_results': r'D:\fsdf_uca_kits\nnunet\results'},
            text=True
        )
        process.wait()
    
    mins = (time.time() - t0) / 60
    n_output = len(list(OUTPUT_FOLDER.glob("*.nii.gz")))
    print(f"Done in {mins:.1f} min")
    print(f"Predictions: {n_output} files")
    print(f"Return code: {process.returncode}")
    
    if process.returncode != 0:
        with open(LOG_FILE) as f:
            print(f"\nLog (last 20 lines):")
            for line in f.readlines()[-20:]:
                print(f"  {line.rstrip()}")

In [ ]:
# ======== Cell 5: Verify Predictions ========
import nibabel as nib
import numpy as np
from pathlib import Path

PRED_DIR = OUTPUT_DIR / "stage1_nnunet_80_20" / "output"
if not PRED_DIR.exists():
    print(f"Prediction directory not found: {PRED_DIR}")
    print("Run Cell 4 commands first!")
else:
    pred_files = sorted(PRED_DIR.glob("*.nii.gz"))
    print(f"Predictions found: {len(pred_files)}")
    
    # Quick quality check on a few cases
    empty_cases = []
    for pf in pred_files[:5]:
        data = nib.load(str(pf)).get_fdata()
        foreground = (data > 0).sum()
        total = data.size
        pct = foreground / total * 100
        print(f"  {pf.name}: shape={data.shape}, foreground={pct:.1f}%")
    
    # Check for empty predictions
    for pf in pred_files:
        data = nib.load(str(pf)).get_fdata()
        if (data > 0).sum() == 0:
            empty_cases.append(pf.stem)
    
    if empty_cases:
        print(f"\nWARNING: {len(empty_cases)} cases with EMPTY predictions:")
        for c in empty_cases:
            print(f"  {c}")
    else:
        print(f"\n✓ All {len(pred_files)} predictions have foreground content")
    
    print(f"\n✓ Predictions verified. Next: NB07c (ROI extraction)")

In [ ]:
import os
# ======== Cell 6: Check Training Log ========
# Run this after training to see best epoch and metrics

from pathlib import Path

results_base = Path(os.environ['nnUNet_results']) / "Dataset001_KiTS"
for config_dir in results_base.iterdir():
    if not config_dir.is_dir():
        continue
    print(f"\n{'='*60}")
    print(f"Configuration: {config_dir.name}")
    print(f"{'='*60}")
    
    fold_dir = config_dir / "fold_0"
    if not fold_dir.exists():
        print("  No fold_0 found")
        continue
    
    # Check checkpoints
    ckpts = sorted(fold_dir.glob("*.pth"))
    print(f"  Checkpoints: {[c.name for c in ckpts]}")
    
    # Read training log
    log_files = sorted(fold_dir.glob("training_log*.txt"))
    if log_files:
        log_file = log_files[-1]  # latest
        with open(log_file) as f:
            lines = f.readlines()
        
        # Find best epoch
        best_lines = [l for l in lines if 'best' in l.lower() or 'ema' in l.lower()]
        print(f"  Log file: {log_file.name} ({len(lines)} lines)")
        if best_lines:
            print(f"  Best epoch info:")
            for bl in best_lines[-5:]:
                print(f"    {bl.strip()}")
        
        # Last few lines
        print(f"  Last 3 lines:")
        for l in lines[-3:]:
            print(f"    {l.strip()}")
    else:
        print(f"  No training log found in {fold_dir}")
    
    # Check progress file
    progress = fold_dir / "progress.png"
    if progress.exists():
        print(f"  ✓ Progress plot: {progress}")